# Proyecto final · Asistente inteligente MDK Tuchtlán
**Maestría en Inteligencia Artificial · Fernanda Gómez Gordillo**

Aplicación web que integra **NLP** (chatbot de reglas + clasificador de intenciones SVM), **visión por computadora** (EfficientNetB0) y un **agente inteligente híbrido** que coordina ambos. Arquitectura: **Streamlit → API REST Flask → agente → modelos**.

| Parte | Qué hace este notebook | Evidencia que genera |
|---|---|---|
| 0 | Obtener el código desde GitHub y verificar versiones | — |
| 1 | Pruebas automáticas (pytest) | salida de pytest |
| 2 | Diagrama de arquitectura | `resultados/arquitectura.png` |
| 3 | Levantar la API Flask y probar errores | `resultados/pruebas_api.csv` |
| 4 | Pruebas funcionales del agente con fotos reales | `resultados/pruebas_agente.csv` |
| 5 | Métricas del clasificador visual | matrices + CSV |
| 6 | Métricas del chatbot | matrices + CSV |
| 7 | Prueba de carga (escalabilidad) | `resultados/prueba_carga.png` |
| 8 | Monitoreo: `/metrics` y registro | `resultados/metrics.json` |
| 9 | Publicar con Cloudflare Tunnel (URL pública) | URL |
| 10 | Descargar todas las evidencias | `evidencias_mdk.zip` |

> ⚙️ No necesita GPU. Ejecuta las celdas **en orden**.

## 0. Código y versiones

In [ ]:
import os
REPO_URL = 'https://github.com/fergordillo880109/chatbot-taekwondo.git'
CARPETA = '/content/chatbot-taekwondo'

if os.path.exists(CARPETA):
    !git -C {CARPETA} pull -q
else:
    !git clone -q {REPO_URL} {CARPETA}
%cd {CARPETA}
!ls

In [ ]:
!pip install -q "streamlit>=1.50,<2"
import sys, keras, tensorflow as tf, sklearn, flask, streamlit
print('Python', sys.version.split()[0], '| TensorFlow', tf.__version__, '| Keras', keras.__version__,
      '| scikit-learn', sklearn.__version__, '| Flask', flask.__version__, '| Streamlit', streamlit.__version__)
if keras.__version__ != '3.13.2':
    print('⚠️ El modelo se guardó con Keras 3.13.2. Si no carga, ejecuta: !pip install -q keras==3.13.2 y reinicia.')

## 1. Pruebas automáticas
Prueban las decisiones del agente (R0–R9), el clasificador de intenciones y la API (validación, códigos HTTP, contexto y métricas) con un modelo visual **simulado**, para que sean rápidas y repetibles.

In [ ]:
!python -m pytest -q tests

## 2. Diagrama de arquitectura

In [ ]:
from graphviz import Digraph
from pathlib import Path
Path('resultados').mkdir(exist_ok=True)

dot = Digraph('Arquitectura', format='png')
dot.attr(rankdir='LR', bgcolor='white', nodesep='0.5', ranksep='0.55')
dot.attr('node', shape='box', style='rounded,filled', fontname='Arial', fontsize='11')
dot.attr('edge', color='#404040', fontname='Arial', fontsize='9')
nodos = {
    'usuario':  ('Familias y alumnos\nnavegador o celular', '#E2F0D9', '#548235'),
    'nube':     ('Streamlit Community Cloud\no Cloudflare Tunnel\nURL pública HTTPS', '#F4CCCC', '#CC0000'),
    'ui':       ('Interfaz Streamlit (app.py)\nchat + foto · session_state\ncontexto de cada usuario', '#D9EAF7', '#2F5597'),
    'api':      ('API REST Flask (api.py)\n/agente  /chat  /predict-image\n/health  /metrics\nvalidación · errores', '#FFF2CC', '#BF9000'),
    'agente':   ('Agente híbrido (agente.py)\npercibir → decidir → actuar\nreglas R0–R9 + memoria', '#EADCF8', '#7030A0'),
    'reglas':   ('Chatbot de reglas\n+ base de conocimiento', '#D0E0E3', '#134F5C'),
    'svm':      ('Clasificador de intenciones\nTF-IDF + LinearSVC (6)', '#D0E0E3', '#134F5C'),
    'cnn':      ('Clasificador visual\nEfficientNetB0 (4 clases)', '#FCE4D6', '#C55A11'),
    'humano':   ('Derivación humana\nWhatsApp de la escuela', '#E2F0D9', '#548235'),
    'monitor':  ('Monitoreo\nlogs/api.log · /metrics\nlatencia · errores · confianza', '#E7E6E6', '#595959'),
}
for k, (t, f, c) in nodos.items():
    dot.node(k, t, fillcolor=f, color=c)
for a, b, l in [('usuario', 'nube', 'HTTPS'), ('nube', 'ui', ''), ('ui', 'api', 'REST · JSON\ntexto + foto + contexto'),
                ('api', 'agente', ''), ('agente', 'reglas', '1. intención'), ('agente', 'svm', '2. si reglas no entienden'),
                ('agente', 'cnn', 'si hay foto'), ('agente', 'humano', 'ambigüedad'), ('api', 'monitor', 'cada solicitud')]:
    dot.edge(a, b, label=l)
dot.render('resultados/arquitectura', cleanup=True)
dot

## 3. API Flask: arranque y manejo de errores
La API carga los modelos **una sola vez** al iniciar. Solo Streamlit queda expuesto a internet; la API escucha en `127.0.0.1:5000`.

In [ ]:
import time, requests, json
!mkdir -p logs && nohup python api.py > logs/api_consola.log 2>&1 &
API = 'http://127.0.0.1:5000'
for _ in range(120):
    try:
        print(json.dumps(requests.get(f'{API}/health', timeout=2).json(), indent=2, ensure_ascii=False)); break
    except requests.exceptions.RequestException:
        time.sleep(1)
else:
    !tail -n 30 logs/api_consola.log

In [ ]:
import io, pandas as pd
from PIL import Image

def png(color=(200, 30, 30)):
    b = io.BytesIO(); Image.new('RGB', (300, 200), color).save(b, format='PNG'); return b.getvalue()

casos = [
    ('Imagen válida',              '/predict-image', dict(files={'file': ('ok.png', png(), 'image/png')}), 200),
    ('Sin archivo',                '/predict-image', dict(), 400),
    ('Extensión .gif',             '/predict-image', dict(files={'file': ('a.gif', b'GIF89a', 'image/gif')}), 400),
    ('Dice PNG pero no es imagen', '/predict-image', dict(files={'file': ('falsa.png', b'texto', 'image/png')}), 400),
    ('Archivo de 6 MB',            '/predict-image', dict(files={'file': ('g.png', b'0' * 6 * 1024 * 1024, 'image/png')}), 413),
    ('Chat correcto',              '/chat', dict(json={'mensaje': '¿Qué días hay clases?'}), 200),
    ('Chat vacío',                 '/chat', dict(json={}), 400),
    ('Mensaje de 600 caracteres',  '/chat', dict(json={'mensaje': 'a' * 600}), 400),
    ('Agente sin texto ni foto',   '/agente', dict(data={'mensaje': ''}), 400),
    ('Contexto corrupto',          '/agente', dict(data={'mensaje': 'hola', 'contexto': '{x'}), 200),
]
filas = []
for nombre, ruta, kw, esperado in casos:
    r = requests.post(API + ruta, **kw)
    cuerpo = r.json()
    filas.append({'caso': nombre, 'endpoint': ruta, 'esperado': esperado, 'obtenido': r.status_code,
                  'correcto': r.status_code == esperado,
                  'respuesta': cuerpo.get('error') or cuerpo.get('clase') or cuerpo.get('decision')})
pruebas_api = pd.DataFrame(filas)
pruebas_api.to_csv('resultados/pruebas_api.csv', index=False)
print(f"{pruebas_api.correcto.sum()}/{len(pruebas_api)} casos correctos")
pruebas_api

## 4. Datos de imágenes y pruebas funcionales del agente
Se usa el mismo `MDK_imagenes_clasificadas.zip` del avance 2 (se busca en tu Drive; si no está, te pedirá subirlo).

**Recomendado:** sube también `fotos_nuevas.zip` con fotos que **nunca** se usaron para entrenar (tomadas por otra persona o en otro lugar), organizadas en `casco/`, `guantes/`, `peto/` y `uniforme/`.

In [ ]:
import zipfile, shutil
from pathlib import Path
from google.colab import drive, files
drive.mount('/content/drive')

def obtener_zip(nombre, ruta_drive, obligatorio=True):
    if Path(ruta_drive).exists():
        shutil.copy(ruta_drive, nombre); return nombre
    print(f'No encontré {ruta_drive}. Sube {nombre}' + ('' if obligatorio else ' (o cancela para omitirlo)'))
    subido = files.upload()
    return nombre if nombre in subido else None

DATOS = Path('/content/mdk_imagenes')
z = obtener_zip('MDK_imagenes_clasificadas.zip', '/content/drive/MyDrive/Chatbot_MDK_Tuchtlan/MDK_imagenes_clasificadas.zip')
zipfile.ZipFile(z).extractall(DATOS)
for parte in ('entrenamiento', 'validacion', 'prueba'):
    print(parte, {p.name: len(list(p.glob('*.jpg'))) for p in sorted((DATOS / parte).iterdir()) if p.is_dir()})

In [ ]:
from evaluacion import casos_agente
from mdk.agente import AgenteMDK
from mdk.nlp import ClasificadorIntenciones
from mdk.vision import ClasificadorVisual
from evaluacion.metricas import fecha_fija

vision = ClasificadorVisual(); assert vision.cargar(), vision.error
agente = AgenteMDK(ClasificadorIntenciones(), vision, fecha=fecha_fija())

def primera(clase):
    return sorted((DATOS / 'prueba' / clase).glob('*.jpg'))[0]

def mezcla():
    """Foto «dudosa»: mitad casco, mitad peto, para provocar baja confianza."""
    a = Image.open(primera('casco')).convert('RGB').resize((224, 224))
    b = Image.open(primera('peto')).convert('RGB').resize((224, 224))
    buf = io.BytesIO(); Image.blend(a, b, 0.5).save(buf, format='JPEG'); return buf.getvalue()

FOTOS = {'casco_claro': primera('casco').read_bytes(), 'peto_claro': primera('peto').read_bytes(),
         'guantes_claro': primera('guantes').read_bytes(), 'uniforme_claro': primera('uniforme').read_bytes(),
         'dudosa': mezcla()}
pruebas_agente = pd.DataFrame(casos_agente.ejecutar(agente, FOTOS.get))
pruebas_agente.to_csv('resultados/pruebas_agente.csv', index=False)
print(f"{pruebas_agente.correcta.sum()}/{len(pruebas_agente)} turnos con la decisión esperada")
pd.set_option('display.max_colwidth', 90)
pruebas_agente[['caso', 'turno', 'entrada', 'imagen', 'esperada', 'reglas', 'confianza', 'correcta', 'respuesta']]

> Si algún turno con foto real no coincide (por ejemplo, la foto «dudosa» sale con confianza alta), **no es un fallo de la prueba**: es un hallazgo para el reporte. Las redes con softmax suelen dar confianzas altas incluso en imágenes ambiguas o fuera de su dominio.

## 5. Métricas del clasificador visual (EfficientNetB0)

In [ ]:
from evaluacion import metricas
from IPython.display import Image as Img, display

resumen_prueba, por_clase, detalle = metricas.evaluar_vision(vision, DATOS / 'prueba', 'prueba')
print(resumen_prueba); display(por_clase); display(Img('resultados/matriz_vision_prueba.png', width=420))

In [ ]:
# Fotos nuevas (generalización). Omite esta celda si aún no las tienes.
z = obtener_zip('fotos_nuevas.zip', '/content/drive/MyDrive/Chatbot_MDK_Tuchtlan/fotos_nuevas.zip', obligatorio=False)
if z:
    NUEVAS = Path('/content/fotos_nuevas'); zipfile.ZipFile(z).extractall(NUEVAS)
    if not (NUEVAS / 'casco').exists():                       # el zip trae una carpeta raíz
        NUEVAS = next(p for p in NUEVAS.iterdir() if p.is_dir())
    resumen_nuevas, por_clase_n, detalle_n = metricas.evaluar_vision(vision, NUEVAS, 'fotos_nuevas')
    print(resumen_nuevas); display(por_clase_n); display(Img('resultados/matriz_vision_fotos_nuevas.png', width=420))
    display(detalle_n[~detalle_n.correcta])

## 6. Métricas del chatbot
- **Clasificador de intenciones:** modelo del avance 2 frente al modelo con el conjunto **ampliado** (retroalimentación del profesor), en la prueba original de 18 frases y en una **prueba independiente** de 60 frases que nunca se usó para entrenar.
- **Sistema completo:** cómo responde el agente (reglas → SVM → humano) a esas 60 frases.

In [ ]:
resumen_int, por_clase_int, errores_int = metricas.evaluar_intenciones()
display(resumen_int); display(por_clase_int); display(errores_int)
display(Img('resultados/matriz_intenciones.png', width=420))

In [ ]:
resumen_chat, tabla_chat = metricas.evaluar_chatbot_completo(agente)
print(resumen_chat)
tabla_chat[~tabla_chat.coherente][['frase', 'etiqueta', 'decision', 'temas', 'respuesta']]

## 7. Prueba de carga (escalabilidad)
Se simulan 1, 5 y 20 usuarios enviando **foto + pregunta** al endpoint `/agente` al mismo tiempo. La latencia es de extremo a extremo: red, validación, agente y CNN.

In [ ]:
carga = metricas.prueba_carga(API, FOTOS['peto_claro'])
display(carga); display(Img('resultados/prueba_carga.png', width=520))

## 8. Monitoreo

In [ ]:
m = requests.get(f'{API}/metrics').json()
json.dump(m, open('resultados/metrics.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(m, indent=2, ensure_ascii=False))
!tail -n 8 logs/api.log

## 9. Publicar la aplicación con Cloudflare Tunnel
Solo se publica **Streamlit (8501)**; la API queda dentro de la máquina. La URL cambia cada vez y deja de funcionar al cerrar Colab: úsala para la demostración y el video. La URL permanente es la de Streamlit Community Cloud (ver README).

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
!AUTO_API=0 nohup streamlit run app.py --server.port 8501 --server.headless true > logs/streamlit.log 2>&1 &
!sleep 6 && curl -s -o /dev/null -w "Streamlit responde con código HTTP %{http_code}\n" http://localhost:8501

import re, subprocess
subprocess.Popen('/content/cloudflared tunnel --url http://localhost:8501 > logs/tunnel.log 2>&1', shell=True)
url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('logs/tunnel.log').read())
    if m:
        url = m.group(0); break
print('Abre la aplicación aquí 👉', url or 'aún no está lista; vuelve a ejecutar esta celda')

## 10. Descargar las evidencias

In [ ]:
!cp logs/api.log resultados/ 2>/dev/null; zip -q -r /content/evidencias_mdk.zip resultados
files.download('/content/evidencias_mdk.zip')

### Capturas que debes tomar tú (para el reporte y la rúbrica)
1. La app abierta en la **URL pública** (Streamlit Cloud) con el estado «Servicio de IA en línea».
2. **Integración final funcionando** (pedido en la retroalimentación del avance 2): una conversación donde
   - escribes «Tengo este, ¿lo puedo llevar?» → el agente pide la foto,
   - adjuntas la foto → responde usando la foto y tu pregunta,
   - preguntas «¿y cuánto cuesta?» → responde con la memoria,
   - y abres «¿Cómo decidió el asistente?».
3. El chatbot respondiendo al menos tres preguntas de texto (horarios, costos, una que resuelva el SVM).
4. La pestaña **Identificar artículo** con una foto y sus probabilidades.
5. La pestaña **Monitoreo** después de usar la app.
6. Un error bien comunicado (por ejemplo, una foto de más de 5 MB o un archivo que no es imagen).
7. El panel de Streamlit Community Cloud (app desplegada) y tu repositorio de GitHub.